In [1]:
import os
import time
import requests
import pandas as pd

In [2]:
# CONFIGURATION
CSV_PATH = "../church-fathers.csv"
OUTPUT_BASE = "../data-2/"


GITHUB_TOKEN = ""  
# GitHub API URLs
GITHUB_API_REPOS = {
    "greek": "https://api.github.com/repos/PerseusDL/canonical-greekLit/contents/data",
    "latin": "https://api.github.com/repos/PerseusDL/canonical-latinLit/contents/data"
}

# GitHub raw content base URLs
GITHUB_RAW_BASE = {
    "greek": "https://raw.githubusercontent.com/PerseusDL/canonical-greekLit/master/data",
    "latin": "https://raw.githubusercontent.com/PerseusDL/canonical-latinLit/master/data"
}

# HTTP SESSION
session = requests.Session()
headers = {
    "User-Agent": "perseus-downloader/1.0 (+https://example.org)",
    "Accept": "application/vnd.github.v3+json"
}
if GITHUB_TOKEN:
    headers["Authorization"] = f"token {GITHUB_TOKEN}"
session.headers.update(headers)


def get_work_folders(ogl_id, language_class):
    """Get list of work folders for an author using GitHub API."""
    api_base = GITHUB_API_REPOS.get(language_class.lower())
    if not api_base:
        print(f"Unknown language class: {language_class}")
        return []
    
    author_api_url = f"{api_base}/{ogl_id}"
    print(f"Fetching work list from API: {author_api_url}")
    
    try:
        r = session.get(author_api_url, timeout=20)
        r.raise_for_status()
        contents = r.json()
        
        # Filter for directories (work folders)
        work_folders = [item['name'] for item in contents if item['type'] == 'dir']
        return work_folders
        
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 404:
            print(f"  Author {ogl_id} not found in repository")
        else:
            print(f"  HTTP Error: {e}")
        return []
    except Exception as e:
        print(f"  Error fetching work folders: {e}")
        return []


def get_xml_files(ogl_id, work_id, language_class):
    """Get list of XML edition files (excluding English translations and metadata) using GitHub API."""
    api_base = GITHUB_API_REPOS.get(language_class.lower())
    work_api_url = f"{api_base}/{ogl_id}/{work_id}"
    
    print(f"  Fetching file list from API: {work_api_url}")
    
    try:
        r = session.get(work_api_url, timeout=20)
        r.raise_for_status()
        contents = r.json()
        
        # Filter for XML files, EXCLUDING:
        # - English translations (-eng)
        # - Metadata files (__cts__.xml)
        xml_files = [
            item['name'] for item in contents 
            if item['type'] == 'file' 
            and item['name'].endswith('.xml')
            and '-eng' not in item['name']
            and item['name'] != '__cts__.xml' 
        ]
        
        return xml_files
        
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 403:
            print(f"  Rate limit exceeded. Please add a GitHub token.")
        else:
            print(f"  Error fetching XML files: {e}")
        return []
    except Exception as e:
        print(f"  Error fetching XML files: {e}")
        return []


def download_xml_file(ogl_id, work_id, filename, output_dir, language_class):
    """Download a single XML file from GitHub raw content."""
    raw_base = GITHUB_RAW_BASE.get(language_class.lower())
    raw_url = f"{raw_base}/{ogl_id}/{work_id}/{filename}"
    output_path = os.path.join(output_dir, filename)
    
    print(f"    Downloading: {filename}")
    
    try:
        r = session.get(raw_url, timeout=30)
        r.raise_for_status()
        
        with open(output_path, 'wb') as fh:
            fh.write(r.content)
        
        print(f"    ✓ Saved: {output_path}")
        return True
    except Exception as e:
        print(f"    ✗ Download failed: {e}")
        return False


def process_author(cf_id, ogl_id, author_name, language_class):
    """Process one author: download all their works."""
    author_dir = os.path.join(OUTPUT_BASE, cf_id)
    os.makedirs(author_dir, exist_ok=True)
    
    print(f"\n{'='*60}")
    print(f"Processing {author_name} ({language_class.upper()})")
    print(f"CF_ID: {cf_id}, OGL_ID: {ogl_id}")
    print(f"{'='*60}")
    
    # Get list of works for this author
    work_folders = get_work_folders(ogl_id, language_class)
    
    if not work_folders:
        print(f"  No works found for {ogl_id} in Perseus repository")
        return
    
    print(f"  Found {len(work_folders)} work(s): {', '.join(work_folders)}")
    
    total_downloaded = 0
    
    # Process each work
    for work_id in work_folders:
        print(f"\n  → Work: {work_id}")
        
        # Get XML files for this work
        xml_files = get_xml_files(ogl_id, work_id, language_class)
        
        if not xml_files:
            print(f"    No edition XML files found for {work_id}")
            continue
        
        print(f"    Found {len(xml_files)} edition file(s)")
        
        # Download each XML file
        for xml_file in xml_files:
            if download_xml_file(ogl_id, work_id, xml_file, author_dir, language_class):
                total_downloaded += 1
            time.sleep(0.3)  # Be nice to GitHub
        
        time.sleep(0.5)  # Pause between works
    
    print(f"\n✓ Completed {author_name}: {total_downloaded} file(s) saved in {author_dir}")


def main():
    # Load CSV
    df = pd.read_csv(CSV_PATH, sep=';', encoding='utf-8')
    
    # Filter rows with ogl_id and class
    df_with_ogl = df[df['ogl_id'].notna() & df['class'].notna()].copy()
    print(f"Found {len(df_with_ogl)} authors with OGL IDs and class.\n")
    
    # Process each author
    for _, row in df_with_ogl.iterrows():
        cf_id = row['CF_ID']
        ogl_id = row['ogl_id']
        name = row['Name']
        language_class = row['class']
        
        process_author(cf_id, ogl_id, name, language_class)
        time.sleep(1)  # Pause between authors
    
    print("All downloads complete!")


if __name__ == "__main__":
    main()

Found 35 authors with OGL IDs and class.


Processing Ignatius von Antiochien (SYRIAN)
CF_ID: CF001, OGL_ID: tlg1443
Unknown language class: syrian
  No works found for tlg1443 in Perseus repository

Processing Polycarpus Smyrnäus (GREEK)
CF_ID: CF002, OGL_ID: tlg1622
Fetching work list from API: https://api.github.com/repos/PerseusDL/canonical-greekLit/contents/data/tlg1622
  Found 1 work(s): tlg001

  → Work: tlg001
  Fetching file list from API: https://api.github.com/repos/PerseusDL/canonical-greekLit/contents/data/tlg1622/tlg001
    No edition XML files found for tlg001

✓ Completed Polycarpus Smyrnäus: 0 file(s) saved in ../data-2/CF002

Processing Athenagoras Atheniensis (GREEK)
CF_ID: CF003, OGL_ID: tlg1205
Fetching work list from API: https://api.github.com/repos/PerseusDL/canonical-greekLit/contents/data/tlg1205
  Author tlg1205 not found in repository
  No works found for tlg1205 in Perseus repository

Processing Theophilus Antiochenus (GREEK)
CF_ID: CF004, OGL_ID: tlg1725
F